# Overview

Building makemore with bigram models.

# Data Loading and EDA

In [ ]:
with open('names.txt', 'r') as file:
    names = file.read().splitlines()

In [ ]:
names[:10]

In [ ]:
len(names)

In [ ]:
min(len(name) for name in names)

In [ ]:
max(len(name) for name in names)

In [ ]:
import itertools

In [ ]:
for name in names[:3]:
    for ch1, ch2 in itertools.pairwise(name):
        print(ch1, ch2)

In [ ]:
b = {}
for name in names:
    chs = ['<S>'] + list(name) + ['<E>']
    for ch1, ch2 in itertools.pairwise(chs):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [ ]:
b.items()

In [ ]:
sorted(b.items(), key=lambda kv: kv[1], reverse=True)[:10]

In [ ]:
import torch

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
chars = sorted(set(''.join(names)))

s_to_i = {s:i + 1 for i, s in enumerate(chars)}
s_to_i['.'] = 0

i_to_s = {i + 1:s for i, s in enumerate(chars)}
i_to_s[0] = "."

In [ ]:
for name in names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in itertools.pairwise(chs):
        idx1, idx2 = s_to_i[ch1], s_to_i[ch2]
        N[idx1, idx2] += 1

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(15, 15))
plt.imshow(N.numpy(), cmap="Blues")
for i in range(27):
    for j in range(27):
        chstr = i_to_s[i] + i_to_s[j]
        plt.text(j, i, chstr, ha='center', va='bottom', color='gray')
        plt.text(j, i, N[i, j].item(), ha='center', va='top', color='gray')
plt.axis('off');

# Table-Based Bigram Model

In [ ]:
# Apply smoothing to prevent -infty
P = (N+1).float()
P /= P.sum(dim=1, keepdim=True)

In [ ]:
g = torch.Generator().manual_seed(42)

def generate_name():
    new_name = ''

    idx = 0
    while True:
        p = P[idx]
        idx = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        # Special character is at index 0
        if idx == 0:
            break
        else:
            new_name += i_to_s[idx]

    return new_name

out = []
for _ in range(5):
    out.append(generate_name())

out

In [ ]:
nll = 0.0
n = 0
for name in names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in itertools.pairwise(chs):
        idx1, idx2 = s_to_i[ch1], s_to_i[ch2]
        p = P[idx1, idx2]
        log_prob = torch.log(p)
        nll -= log_prob
        n += 1

print(f"Average negative log likelihood: {nll / n}")

# Neural Net-Based Bigram Model (End-to-End)

Steps:
1. Create dataset
2. Create simple neural net
3. Train with gradient descent
4. Sample from neural net to get names

The neural network here is just a linear layer (followed by softmax). With this setup, the best performance it can possibly achieve is that of the table-based model, so we should expect the model to converge to the performance of the table-based model with gradient descent.

In [ ]:
with open('names.txt', 'r') as file:
    names = file.read().splitlines()

xs, ys = [], []
for name in names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in itertools.pairwise(chs):
        idx1, idx2 = s_to_i[ch1], s_to_i[ch2]
        xs.append(idx1)
        ys.append(idx2)

xs, ys = torch.tensor(xs), torch.tensor(ys)

In [ ]:
import torch.nn.functional as F

In [ ]:
X_enc = F.one_hot(xs, num_classes=27).float()

In [ ]:
g = torch.Generator().manual_seed(42)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
y_enc = F.one_hot(ys, num_classes=27).bool()

num_epochs = 200
for epoch in range(num_epochs):
    W.grad = None
    # Forward pass
    logits = X_enc @ W
    probs = logits.exp() / logits.exp().sum(dim=1, keepdims=True)
    # Compute loss
    loss = -probs[y_enc].log().mean()
    # Backward pass
    loss.backward()
    with torch.no_grad():
        W -= 50 * W.grad

    print(f"Epoch: {epoch + 1}, loss: {loss}")